# ARCSIX single-granule presentation figures

Presentation companion derived from `analysis_arcsix_single_granule.ipynb`. The analysis notebook remains the place to prepare or change the retrieval; this notebook reads its **completed** raw, companion, and reference caches for **20240610T154205 (June 10, 2024, 15:42–15:47 UTC)**. No LUT recomputation or airborne download is needed.

Outputs: three aligned 16:9 dashboard PNGs, three matching modal-phase map crops, a true-color context dashboard, a matching true-color map crop, and a JSON record of settings and file hashes. The original full-granule histogram population is retained. The fixed 265 K/global ocean-LUT approximation and provisional phase labels remain unchanged.

Run in order. The first RGB extraction reads three visible channels from the **same OCI L1C V3 granule**, then stores a small reflectance cache. Subsequent exports are local. RGB is a gamma-enhanced TOA view, not an atmospherically corrected surface product or independent phase truth.

## 1. Locate the repository and select your existing Arctic config

Open this notebook inside the cloned repository. Repository discovery below uses the notebook working directory and its parents. If your kernel starts elsewhere (for example your home folder), set `REPO_OVERRIDE` to your local clone. `PACE_CONFIG`, if set, takes priority; otherwise use `config/arcsix.local.toml`. This cell does not create, move, or modify either local TOML file.

In [ ]:
from pathlib import Path
import os

REPO_OVERRIDE = None  # Example on CryoCloud: "~/PACE_spectropolarimetric_phase/PACE-specpol-phase"
if REPO_OVERRIDE:
    REPO = Path(REPO_OVERRIDE).expanduser().resolve()
else:
    REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src/pace_specpol").is_dir()), None)
if REPO is None or not (REPO / "pyproject.toml").is_file():
    raise FileNotFoundError("Set REPO_OVERRIDE to your local PACE-specpol-phase clone.")
CONFIG = Path(os.environ.get("PACE_CONFIG", str(REPO / "config/arcsix.local.toml"))).expanduser().resolve()
if not CONFIG.is_file():
    raise FileNotFoundError(f"Select your existing Arctic paths TOML; not found: {CONFIG}")
print("Repository:", REPO)
print("Local configuration:", CONFIG)

## 2. Install after a fresh CryoCloud environment restart

Run this cell after the server image has reset, or when installing this branch for the first time. The editable install uses this kernel's Python. It does not modify your TOML files. Restart the **kernel** if prompted, then rerun the setup cells; skip the install cell if the package is already installed from this clone.

In [ ]:
%pip install -e "{REPO}[validation]"

In [ ]:
import json
from IPython.display import display, Image
from pace_specpol.paths import load_paths
from pace_specpol.workflow import VariantReference
from pace_specpol.validation.aggregation import IndexConfig, build_index
from pace_specpol.validation.vis_dashboard import dashboard
from pace_specpol.validation.vis_truecolor import presentation_inputs, cache_rgb
from pace_specpol.validation.vis_presentation import export_presentation

PATHS = load_paths(CONFIG)
TARGET_STAMP = "20240610T154205"
EXPERIMENT = "arcsix_20240610T154205_global_ocean3_265K_v1"
RUN = f"{EXPERIMENT}_pilot1"
PAIR_CACHE = PATHS["pair_cache"]
WORK_ROOT = PATHS["work_root"]
COMPANION_DIR = WORK_ROOT / RUN / "companions"
REFERENCE_DIR = WORK_ROOT / RUN / "references"

# Independent presentation products; do not edit any scientific cache manifest.
RGB_CACHE = WORK_ROOT / RUN / "presentation_inputs" / "oci_visible_rgb_v1.nc"
EXPORT_LABEL = "threshold_demo_v1"  # Change for a new set of figures.
EXPORT_DIR = PATHS["export_root"] / RUN / "presentation" / EXPORT_LABEL
OVERWRITE = False  # Explicitly True only to replace an earlier figure set.

oci_descriptor, match_settings = presentation_inputs(
    PAIR_CACHE, COMPANION_DIR, REFERENCE_DIR, TARGET_STAMP)
print("Verified completed single-granule cache chain.")
print("OCI source:", oci_descriptor["name"])
print("Reference cache:", REFERENCE_DIR)
print("Figure destination:", EXPORT_DIR)

## 3. Choose the three presentation states

Default: normalized ratio, OCI 2260 CER/COT, each mode's own valid population; LI stays at 0.3. **1.10, 1.27, and 1.45 are illustrative thresholds, not newly calibrated OCI boundaries.** Change `THRESHOLDS` on the 0.01 grid. Keep the variant, population and ratio space fixed within a sequence so the comparison changes only the ratio threshold.

The mean LI and mean ratio values do not change with that threshold. The ratio map's diverging color scale **recenters** with it, just as in the interactive dashboard. Full-granule histogram counts stay fixed; the polar extent affects only map display.

In [ ]:
SELECTED_VARIANT = "oci_2260"  # oci_2130, harp2_2260, harp2_2130
SELECTED_POPULATION = "own"    # common
RATIO_MODE = "normalized"      # corrected or raw require separate interpretation
THRESHOLDS = (1.10, 1.27, 1.45)
CONTEXT_THRESHOLD = 1.27

INDEX_CONFIG = IndexConfig(resolution=0.1, ratio_min=0.5, ratio_max=3.5,
                           ratio_step=0.01, li_threshold=0.3)
MAP_OPTIONS = dict(map_extent=(-85, -10, 58, 87), map_framing="granule",
                   graticules=True, preview_width=1400, min_count=1,
                   ratio_clim=(0.5, 2.0), li_clim=(-0.5, 4.0))
index = build_index(REFERENCE_DIR,
                    reference=VariantReference(SELECTED_VARIANT, SELECTED_POPULATION, RATIO_MODE),
                    config=INDEX_CONFIG)
for threshold in (*THRESHOLDS, CONTEXT_THRESHOLD):
    index.threshold_index(threshold)
print("Eligible samples:", index.metadata["totals"]["indexed_samples"])
print("Reference selection:", index.metadata["reference"])

## 4. Optional interactive preview

This preview reads the local index. The batch export below uses the explicit `THRESHOLDS` and `MAP_OPTIONS` above, **not the slider's last position**. The preview's save button is separate from the aligned batch export.

In [ ]:
SHOW_INTERACTIVE = False
if SHOW_INTERACTIVE:
    preview = dashboard(index, threshold=CONTEXT_THRESHOLD, layout="slide",
                        output_dir=EXPORT_DIR / "interactive", **MAP_OPTIONS)

## 5. Load/cache OCI true color

Use the nearest available OCI bands to **645 / 555 / 469 nm** for red / green / blue. The reader reports the actual wavelengths and converts radiance using the source solar irradiances, Earth–Sun distance, and solar zenith. All channels come from one valid view per grid cell, preferring the smallest absolute time offset from nadir. Geometry limits and the OCI QC policy come from the paired-cache settings; missing QC is reported rather than described as valid quality assurance.

This is the **full visible 5 km L1C scene**, without HARP2 cloud/LI or liquid-reference screening. The selected RGB view can differ from the SWIR-selected view where band availability differs. Thus RGB coverage can exceed phase-map coverage. No extra cloud parallax or advection correction is applied.

On CryoCloud, leave `LOCAL_OCI_SOURCE=None`: the first call logs into Earthdata and reads the exact source via S3. A later call loads `RGB_CACHE` without login. Outside AWS, optionally supply an already downloaded copy of that exact OCI L1C file. Only the extracted RGB reflectances are cached here; changes to gamma or thresholds do not download it again.

In [ ]:
LOCAL_OCI_SOURCE = None  # Optional absolute path to the exact original OCI L1C .nc file.
rgb = cache_rgb(oci_descriptor, RGB_CACHE, match_settings, local_source=LOCAL_OCI_SOURCE)
print("RGB pixels:", rgb.sizes["pixel"])
print("Actual RGB wavelengths (nm, per view):", rgb.attrs["actual_wavelengths_nm"])
print("OCI QC policy:", rgb.attrs["qc_policy"])
print("RGB pixels with unknown QC:", rgb.attrs["unknown_qc_pixels"])
print("Local visible-reflectance cache:", RGB_CACHE)

## 6. Export aligned figures

The default 240 dpi canvas is **3200 × 1800 pixels (16:9)**. Every threshold reuses the same Figure and map axes. Map-only PNGs share the same exact pixel crop, projection, and geographic limits as the upper-right dashboard map. Do not independently crop or stretch them in PowerPoint.

RGB uses one fixed stretch for all channels: `clip((rho - black)/(white - black), 0, 1) ** (1/gamma)`. It is an enhanced approximate true-color composite, not a colorimetric rendering. The nearest L1C center is used only within 4 km; farther regions remain white. This display interpolation does not increase native resolution. The context dashboard uses `CONTEXT_THRESHOLD` for its unchanged histogram and lower maps.

In [ ]:
export_record = export_presentation(
    index, EXPORT_DIR, thresholds=THRESHOLDS, context_threshold=CONTEXT_THRESHOLD,
    rgb=rgb, dpi=240, overwrite=OVERWRITE,
    rgb_black=0.0, rgb_white=1.0, rgb_gamma=2.2, rgb_max_distance_km=4.0,
    **MAP_OPTIONS,
)
print("Full dashboard size:", export_record["canvas_pixels"])
print("Matching map size:", export_record["map_pixels"])
print("Files:")
for item in export_record["files"]:
    print(EXPORT_DIR / item["name"])
print("Settings/provenance:", EXPORT_DIR / "presentation_manifest.json")

In [ ]:
# Preview exported files without changing their saved dimensions.
display(Image(filename=str(EXPORT_DIR / "dashboard_truecolor_context.png"), width=1200))
for threshold in THRESHOLDS:
    tag = f"{threshold:.2f}".replace(".", "p")
    display(Image(filename=str(EXPORT_DIR / f"dashboard_ratio_{tag}.png"), width=1200))

## Use in PowerPoint

**Simple sequence:** use a 16:9 deck. Insert `dashboard_truecolor_context.png`, set the image to fill the slide, and duplicate the slide for each `dashboard_ratio_*.png`. Replace the picture while retaining the exact size and position. A short **Fade** transition provides a clear reveal without requiring an animation or a generated video.

**Clickable demonstration:** duplicate three threshold slides, put the same three labeled buttons on each, and link each button to its corresponding threshold slide. Reserve the same area for buttons on every slide and keep all dashboard images identically positioned. These are links among three pre-rendered states, not a live recalculation.

**Map-only reveal:** place `truecolor_map.png` and `phase_map_ratio_*.png` at exactly the same size and position on successive slides. They are cut from the same map box, without titles or legends; add your own consistent labels. For a reveal within a full dashboard, the JSON file records `map_crop_box_pixels` as left/top/right/bottom pixel coordinates on the full canvas.

White areas have different meanings: RGB lacks visible coverage there; phase maps can also be white because of scientific screening, insufficient counts, or tied modal phase. The RGB image is scene context, not an independent confirmation of liquid or ice.